# NB12 — ablation A8: preprocessing without ICA + reference cells on the confirmation folds (CPU)

Re-downloads ds004504, runs the fixed preamble with ICA/ICLabel skipped (`preprocess.ica_method: none`), rebuilds the cache, trains P3 x spatial and P3 x hybrid on the 25 confirmation splits, and repeats the fixed-threshold PSWE statistics (eye movements are slow and could masquerade as slow-wave events).

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess,dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_external.py tests/test_model_train.py', log=f'{WORK}/pytest_abl.txt')

In [ ]:
C = f'{PY} --config configs/ablation_a8_no_ica.yaml'
sh(f'{C} download --dest /tmp/ds004504', log=f'{WORK}/download.log')
shutil.copy('/tmp/ds004504/participants.tsv', f'{WORK}/participants.tsv')
sh(f'{C} preprocess --bids /tmp/ds004504 --out /tmp/preproc --n-jobs 4', log=f'{WORK}/preprocess.log')
shutil.copy('/tmp/preproc/preprocess_log.jsonl', f'{WORK}/preprocess_log.jsonl')

In [ ]:
sh(f'{C} cache --preproc /tmp/preproc --participants {WORK}/participants.tsv --out /tmp/cache', log=f'{WORK}/cache.log')
sh(f'{C} qc --cache /tmp/cache --logs /tmp/preproc/preprocess_log.jsonl --out {WORK}/qc_A8_no_ica.json --no-fail')
sh(f'{C} pswe-stats --cache /tmp/cache --participants {WORK}/participants.tsv --out {WORK}/rq6_no_ica', log=f'{WORK}/pswe_stats.txt')

In [ ]:
sh(f'{C} grid --cache /tmp/cache --folds {SRC}/splits/folds_ds004504_confirm.json --pipelines P3 --edges spatial --tag A8_no_ica@P3xspatial --out {WORK}/ablations_confirm.jsonl --max-hours 5', log=f'{WORK}/A8_no_ica_spatial.log')
sh(f'{C} grid --cache /tmp/cache --folds {SRC}/splits/folds_ds004504_confirm.json --pipelines P3 --edges hybrid --tag A8_no_ica@P3xhybrid --out {WORK}/ablations_confirm.jsonl --max-hours 5', log=f'{WORK}/A8_no_ica_hybrid.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)